In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report


In [2]:
# 🧱 Step 1 — Load and Clean Data
# Load dataset
df = pd.read_parquet("D:\LPA_MTech_Project\Enriched_Datasets\SupremeCourt_Combined_2020_2025_enriched.parquet")

# Keep only rows with labels
df = df.dropna(subset=['verdict_label']).copy()

# Basic text cleaning
df['clean_text'] = (
    df['text'].astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)

# Select key features
text_col = 'clean_text'
num_cols = ['bench_size', 'sentiment_score', 'num_citations',
            'pet_vs_resp_ratio', 'num_unique_acts']
cat_col = 'case_type'
target_col = 'verdict_label'


In [3]:
# 🧮 Step 2 — Encode Case Type + Scale Numerics
# One-hot encode case_type
df = pd.get_dummies(df, columns=[cat_col], drop_first=True)

# Numeric features
X_num = df[num_cols].fillna(0).values
scaler = StandardScaler()
X_num = scaler.fit_transform(X_num)

# Case type one-hot features
X_cat = df.filter(like='case_type_').values


In [4]:
# 🧠 Step 3 — Text Features (TF-IDF + SVD)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1,2), stop_words='english')
X_text_tfidf = tfidf.fit_transform(df[text_col])

# Dimensionality reduction
svd = TruncatedSVD(n_components=300, random_state=42)
X_text_reduced = svd.fit_transform(X_text_tfidf)



In [5]:
# ⚙️ Step 4 — Combine All Features
# Combine text (300-d) + numeric + categorical
X_combined = np.hstack((X_text_reduced, X_num, X_cat))
y = df[target_col].astype(int).values

print("Final Feature Shape:", X_combined.shape)


Final Feature Shape: (4061, 309)


In [6]:
# 🧩 Step 5 — Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_combined, y, test_size=0.2, stratify=y, random_state=42
)


In [7]:
# ⚡ Step 6 — Prepare PyTorch Data (GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


Using device: cuda


In [8]:
# 🧠 Step 7 — Define Balanced PyTorch Model
class VerdictNetV2(nn.Module):
    def __init__(self, input_dim):
        super(VerdictNetV2, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)

model = VerdictNetV2(X_train.shape[1]).to(device)


In [9]:
# ⚖️ Step 8 — Handle Class Imbalance
# Compute class weights
class_counts = np.bincount(y_train)
weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)



In [10]:
# 🧩 Step 9 — Train Model
epochs = 15
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(X_batch)
        loss = criterion(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/15 - Loss: 0.6641
Epoch 2/15 - Loss: 0.6061
Epoch 3/15 - Loss: 0.5299
Epoch 4/15 - Loss: 0.4858
Epoch 5/15 - Loss: 0.4574
Epoch 6/15 - Loss: 0.4317
Epoch 7/15 - Loss: 0.3861
Epoch 8/15 - Loss: 0.3578
Epoch 9/15 - Loss: 0.3360
Epoch 10/15 - Loss: 0.3075
Epoch 11/15 - Loss: 0.2828
Epoch 12/15 - Loss: 0.2687
Epoch 13/15 - Loss: 0.2467
Epoch 14/15 - Loss: 0.2218
Epoch 15/15 - Loss: 0.2142


In [11]:
# 🧪 Step 10 — Evaluate
model.eval()
y_pred_list = []

with torch.no_grad():
    for X_batch, _ in test_loader:
        X_batch = X_batch.to(device)
        preds = model(X_batch)
        y_pred = torch.argmax(preds, dim=1)
        y_pred_list.extend(y_pred.cpu().numpy())

acc = accuracy_score(y_test, y_pred_list)
print("\nTest Accuracy:", acc)
print("\nClassification Report:\n", classification_report(y_test, y_pred_list))



Test Accuracy: 0.6900369003690037

Classification Report:
               precision    recall  f1-score   support

           0       0.39      0.43      0.41       201
           1       0.81      0.77      0.79       612

    accuracy                           0.69       813
   macro avg       0.60      0.60      0.60       813
weighted avg       0.70      0.69      0.70       813



In [12]:
# 💾 Step 11 — Save Artifacts
torch.save(model.state_dict(), "verdict_predictor_v2.pt")

import joblib
joblib.dump(tfidf, "tfidf_vectorizer_v2.pkl")
joblib.dump(svd, "svd_model_v2.pkl")
joblib.dump(scaler, "num_scaler_v2.pkl")


['num_scaler_v2.pkl']